# 01. 객체 검출·분할 Metric 기초

## 목표
Bounding box IoU, binary mask IoU, precision, recall과 F1을 직접 계산합니다. 논문의 TP 조건인 class 일치와 IoU > 0.5를 작은 예제로 확인합니다.

In [ ]:
def box_iou(a, b):
    # box 형식: (x1, y1, x2, y2)
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0

ground_truth = (10, 10, 30, 30)
prediction = (12, 8, 32, 29)
print(f"box IoU: {box_iou(ground_truth, prediction):.3f}")

In [ ]:
def mask_iou(a, b):
    flat_pairs = zip((v for row in a for v in row), (v for row in b for v in row))
    intersection = union = 0
    for left, right in flat_pairs:
        intersection += bool(left) and bool(right)
        union += bool(left) or bool(right)
    return intersection / union if union else 0.0

mask_gt = [
    [0, 1, 1, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0],
]
mask_pred = [
    [0, 1, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 0, 0],
]
print(f"mask IoU: {mask_iou(mask_gt, mask_pred):.3f}")

In [ ]:
def classification_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    penalized_iou = 3.2 / (tp + fp + fn)  # 예시 TP IoU 합이 3.2라고 가정
    return precision, recall, f1, penalized_iou

precision, recall, f1, penalized_iou = classification_metrics(4, 2, 1)
print(f"precision={precision:.3f}, recall={recall:.3f}, F1={f1:.3f}")
print(f"penalized IoU={penalized_iou:.3f}")

## 연습

1. IoU가 정확히 0.5인 예측은 논문의 `> 0.5` 조건에서 TP인지 확인하세요.
2. 같은 ground truth에 예측 두 개가 겹칠 때 하나만 TP가 되도록 matching을 구현하세요.
3. Class mismatch를 FP 하나와 FN 하나로 기록해 metric 변화를 비교하세요.